# YOLO11n Wine Label Detector

Production-like workflow for:

1. manual annotation bootstrap;
2. YOLO dataset validation and visual inspection;
3. YOLO11n training on Apple Silicon MPS, NVIDIA CUDA, or CPU;
4. validation metrics and error-oriented visualization;
5. robust batch cropping for the downstream DINO retrieval dataset;
6. active-learning selection and inference benchmarking.

The notebook detects one class: `wine_label` (`class_id = 0`). It is safe to run top-to-bottom: long-running stages are disabled by flags until explicitly enabled.

> The intended target is the **main front wine label**, not the whole bottle. Do not manually annotate all 45,000 images; bootstrap with 100–150 images and iterate.

### 1.1 Project environment

All runtime and Jupyter dependencies are declared in the project-root `requirements.txt` and installed into `PROJECT_ROOT/.venv`. Select the **VIscaNEr YOLO (.venv)** kernel. The optional repair cell below installs from that single requirements file; it does not touch the system Python environment.

In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT_FOR_ENV = Path("/Users/konstantinmelnikov/Desktop/work/VIscaNEr")
REQUIREMENTS_FILE = PROJECT_ROOT_FOR_ENV / "requirements.txt"
INSTALL_DEPENDENCIES = False

PACKAGE_TO_MODULE = {
    "torch": "torch",
    "torchvision": "torchvision",
    "ultralytics": "ultralytics",
    "opencv-python": "cv2",
    "Pillow": "PIL",
    "matplotlib": "matplotlib",
    "pandas": "pandas",
    "numpy": "numpy",
    "PyYAML": "yaml",
    "tqdm": "tqdm",
    "jupyterlab": "jupyterlab",
    "ipykernel": "ipykernel",
    "nbformat": "nbformat",
    "nbclient": "nbclient",
    "ipywidgets": "ipywidgets",
}

missing_packages = [
    package
    for package, module in PACKAGE_TO_MODULE.items()
    if importlib.util.find_spec(module) is None
]

print("Python executable:", sys.executable)
print("Requirements:", REQUIREMENTS_FILE)
if INSTALL_DEPENDENCIES:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-r", str(REQUIREMENTS_FILE)]
    )
elif missing_packages:
    print("Missing packages:", ", ".join(missing_packages))
    print("Select the VIscaNEr YOLO (.venv) kernel, or set INSTALL_DEPENDENCIES = True.")
else:
    print("All requirements are importable in this kernel.")

### 1.2 Imports, reproducibility, and accelerator selection

Device priority is CUDA → MPS → CPU. CUDA uses device index `0`; MPS and CPU use string device names accepted by Ultralytics.

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import platform
import random
import shutil
import sys
import time
import traceback
from pathlib import Path
from pprint import pprint
from typing import Any, Iterable, Optional

import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import pandas as pd
import torch
import ultralytics
import yaml
from IPython.display import Image as IPythonImage
from IPython.display import display
from PIL import Image, ImageOps
from tqdm.auto import tqdm
from ultralytics import YOLO

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    DEVICE = 0
    DEVICE_NAME = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
    DEVICE_NAME = "mps"
else:
    DEVICE = "cpu"
    DEVICE_NAME = "cpu"

plt.rcParams.update({
    "figure.figsize": (12, 8),
    "figure.dpi": 110,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
})

print(f"OS: {platform.platform()}")
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"Ultralytics: {ultralytics.__version__}")
print(f"Device: {DEVICE_NAME} ({DEVICE!r})")
print(f"MPS available: {hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

### 1.3 Project paths and execution flags

All paths are derived from `PROJECT_ROOT`. Set `BULK_INPUT_DIR` to the real directory containing the large source dataset before enabling bulk crop. Generated directories are created with `exist_ok=True`; no existing data is deleted.

In [ ]:
PROJECT_ROOT = Path("/Users/konstantinmelnikov/Desktop/work/VIscaNEr")
DATASETS_ROOT = PROJECT_ROOT / "datasets"

# The user-provided two-class annotation export. It is read-only in this workflow:
# source class 0 = label, source class 1 = bottle.
SOURCE_ANNOTATED_DATASET_ROOT = DATASETS_ROOT / "wine_labels_yolo_500"
SOURCE_DATASET_YAML = SOURCE_ANNOTATED_DATASET_ROOT / "data.yaml"

# Single-class training view generated from the source dataset without changing it.
YOLO_DATASET_ROOT = DATASETS_ROOT / "yolo_label_detector"
PREPARED_DATASET_ROOT = YOLO_DATASET_ROOT / "manual_annotation"
MANUAL_ANNOTATION_ROOT = PREPARED_DATASET_ROOT
RAW_IMAGES_DIR = PREPARED_DATASET_ROOT / "raw_images"
TRAIN_IMAGES_DIR = PREPARED_DATASET_ROOT / "images" / "train"
VAL_IMAGES_DIR = PREPARED_DATASET_ROOT / "images" / "val"
TRAIN_LABELS_DIR = PREPARED_DATASET_ROOT / "labels" / "train"
VAL_LABELS_DIR = PREPARED_DATASET_ROOT / "labels" / "val"
INFERENCE_INPUT_DIR = YOLO_DATASET_ROOT / "inference_input"
CROPS_ROOT = YOLO_DATASET_ROOT / "crops"
SUCCESSFUL_CROPS_DIR = CROPS_ROOT / "successful"
LOW_CONFIDENCE_CROPS_DIR = CROPS_ROOT / "low_confidence"
FAILED_CROPS_DIR = CROPS_ROOT / "failed"
PREDICTIONS_DEBUG_DIR = YOLO_DATASET_ROOT / "predictions_debug"
DATASET_YAML = YOLO_DATASET_ROOT / "dataset.yaml"
MODEL_OUTPUT_DIR = PROJECT_ROOT / "models" / "yolo_label_detector"
RUNS_ROOT = PROJECT_ROOT / "runs" / "yolo_label_detector"

# Unified flat directory built from the two 45k bottle-image source parts.
BULK_INPUT_DIR = DATASETS_ROOT / "bottle_images_45k"

# Long-running or output-producing stages are opt-in.
RUN_TRAINING = False
RUN_VALIDATION = False
RUN_VALIDATION_VISUALIZATION = False
RUN_SAMPLE_CROP = False
RUN_BULK_CROP = False
RUN_BENCHMARK = False
SAVE_DEBUG = False  # successful detections; low-confidence and failed always get debug output

DIRECTORIES = [
    RAW_IMAGES_DIR,
    TRAIN_IMAGES_DIR,
    VAL_IMAGES_DIR,
    TRAIN_LABELS_DIR,
    VAL_LABELS_DIR,
    INFERENCE_INPUT_DIR,
    SUCCESSFUL_CROPS_DIR,
    LOW_CONFIDENCE_CROPS_DIR,
    FAILED_CROPS_DIR,
    PREDICTIONS_DEBUG_DIR,
    MODEL_OUTPUT_DIR,
    RUNS_ROOT,
]
for directory in DIRECTORIES:
    directory.mkdir(parents=True, exist_ok=True)

if not SOURCE_ANNOTATED_DATASET_ROOT.is_dir():
    raise FileNotFoundError(f"Annotated source dataset not found: {SOURCE_ANNOTATED_DATASET_ROOT}")

dataset_config = {
    "path": str(PREPARED_DATASET_ROOT.resolve()),
    "train": "images/train",
    "val": "images/val",
    "names": {0: "wine_label"},
}
yaml_text = yaml.safe_dump(dataset_config, sort_keys=False, allow_unicode=True)
if not DATASET_YAML.exists() or DATASET_YAML.read_text(encoding="utf-8") != yaml_text:
    DATASET_YAML.write_text(yaml_text, encoding="utf-8")

print(f"Project root: {PROJECT_ROOT}")
print(f"Annotated source (unchanged): {SOURCE_ANNOTATED_DATASET_ROOT}")
print(f"Prepared one-class dataset: {PREPARED_DATASET_ROOT}")
print(f"Dataset YAML: {DATASET_YAML}")
print(DATASET_YAML.read_text(encoding="utf-8"))

## 2. Prepare the existing 500-image annotation export

`datasets/wine_labels_yolo_500/` contains 400 train and 100 validation images with two source classes: `0 = label` and `1 = bottle`. The target detector is deliberately one-class, so this idempotent step creates a derived training view under `datasets/yolo_label_detector/manual_annotation/`:

- images are symlinked to avoid duplication;
- only source class `0` rows are retained and named `wine_label`;
- source class `1` (`bottle`) is ignored;
- the original 500-image dataset is never modified or deleted.

Re-run this preparation cell whenever the source annotations change.

In [ ]:
SOURCE_LABEL_CLASS_ID = 0
SOURCE_BOTTLE_CLASS_ID = 1
SUPPORTED_IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}


def prepare_single_class_dataset(
    source_root: Path = SOURCE_ANNOTATED_DATASET_ROOT,
    output_root: Path = PREPARED_DATASET_ROOT,
    source_label_class_id: int = SOURCE_LABEL_CLASS_ID,
) -> pd.DataFrame:
    """Create a reproducible one-class YOLO view while preserving the source dataset."""
    source_root = Path(source_root).resolve()
    output_root = Path(output_root).resolve()
    if source_root == output_root:
        raise ValueError("Source and output dataset roots must be different.")

    summary_rows: list[dict[str, int | str]] = []
    problems: list[str] = []

    for split in ("train", "val"):
        source_images_dir = source_root / "images" / split
        source_labels_dir = source_root / "labels" / split
        output_images_dir = output_root / "images" / split
        output_labels_dir = output_root / "labels" / split
        output_images_dir.mkdir(parents=True, exist_ok=True)
        output_labels_dir.mkdir(parents=True, exist_ok=True)

        source_images = sorted(
            path for path in source_images_dir.rglob("*")
            if path.is_file() and path.suffix.lower() in SUPPORTED_IMAGE_EXTENSIONS
        )
        linked_images = 0
        reused_images = 0
        written_labels = 0
        wine_label_boxes = 0
        ignored_bottle_boxes = 0

        for source_image in source_images:
            relative_image = source_image.relative_to(source_images_dir)
            source_label = source_labels_dir / relative_image.with_suffix(".txt")
            output_image = output_images_dir / relative_image
            output_label = output_labels_dir / relative_image.with_suffix(".txt")
            output_image.parent.mkdir(parents=True, exist_ok=True)
            output_label.parent.mkdir(parents=True, exist_ok=True)

            if not source_label.exists():
                problems.append(f"Missing source label: {source_label}")
                continue

            if output_image.exists() or output_image.is_symlink():
                if output_image.is_symlink() and output_image.resolve() != source_image.resolve():
                    problems.append(f"Wrong existing symlink: {output_image}")
                    continue
                reused_images += 1
            else:
                output_image.symlink_to(source_image.resolve())
                linked_images += 1

            filtered_rows: list[str] = []
            for line_number, raw_line in enumerate(source_label.read_text(encoding="utf-8").splitlines(), 1):
                line = raw_line.strip()
                if not line:
                    continue
                parts = line.split()
                if len(parts) != 5:
                    problems.append(f"{source_label}:{line_number}: expected 5 fields")
                    continue
                try:
                    class_value = float(parts[0])
                    coordinates = [float(value) for value in parts[1:]]
                except ValueError:
                    problems.append(f"{source_label}:{line_number}: non-numeric YOLO row")
                    continue
                if not class_value.is_integer():
                    problems.append(f"{source_label}:{line_number}: class id is not an integer")
                    continue

                class_id = int(class_value)
                if class_id == source_label_class_id:
                    filtered_rows.append("0 " + " ".join(f"{value:.6f}" for value in coordinates))
                    wine_label_boxes += 1
                elif class_id == SOURCE_BOTTLE_CLASS_ID:
                    ignored_bottle_boxes += 1
                else:
                    problems.append(f"{source_label}:{line_number}: unexpected class {class_id}")

            filtered_text = "\n".join(filtered_rows)
            if filtered_text:
                filtered_text += "\n"
            if not output_label.exists() or output_label.read_text(encoding="utf-8") != filtered_text:
                output_label.write_text(filtered_text, encoding="utf-8")
                written_labels += 1

        summary_rows.append({
            "split": split,
            "source_images": len(source_images),
            "new_image_links": linked_images,
            "reused_images": reused_images,
            "updated_label_files": written_labels,
            "wine_label_boxes": wine_label_boxes,
            "ignored_bottle_boxes": ignored_bottle_boxes,
        })

    if problems:
        preview = "\n".join(problems[:30])
        raise RuntimeError(f"Dataset preparation found {len(problems)} problem(s):\n{preview}")

    summary = pd.DataFrame(summary_rows).set_index("split")
    display(summary)
    print("Source dataset was read only; prepared one-class labels are ready.")
    return summary


PREPARATION_REPORT = prepare_single_class_dataset()

## 3. Manual annotation and future iterations

The current bootstrap is already available in `datasets/wine_labels_yolo_500/`. For new active-learning rounds:

1. Put new source photos into `datasets/yolo_label_detector/manual_annotation/raw_images/`.
2. Annotate tight boxes around the main front label as class `0 = wine_label`.
3. Correct low-confidence and failed detections first.
4. Merge reviewed examples into a new source-dataset version, then re-run the preparation cell.

Do not manually annotate the full 45k-image collection. Keep near-duplicate frames from the same bottle in one split to avoid leakage.

## 4. Validate the prepared YOLO dataset

The validator checks paired image/label files, one-class format, normalized coordinates, empty labels, orphan labels, and malformed rows. Missing labels and malformed rows are errors. Empty labels are reported as warnings because YOLO can use explicit background images, but confirm they are intentional.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}


def list_images(directory: Path, recursive: bool = True) -> list[Path]:
    '''Return supported image files in deterministic order.'''
    directory = Path(directory)
    if not directory.exists():
        return []
    iterator = directory.rglob("*") if recursive else directory.glob("*")
    return sorted(
        path for path in iterator
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )


def _parse_yolo_label_file(label_path: Path) -> tuple[list[tuple[float, ...]], list[str], bool]:
    '''Parse one label file and return valid boxes, issues, and empty-label flag.'''
    issues: list[str] = []
    boxes: list[tuple[float, ...]] = []
    try:
        text = label_path.read_text(encoding="utf-8")
    except Exception as exc:
        return boxes, [f"{label_path}: cannot read label ({exc})"], False

    nonempty_lines = [(line_number, line.strip()) for line_number, line in enumerate(text.splitlines(), 1) if line.strip()]
    is_empty = len(nonempty_lines) == 0

    for line_number, line in nonempty_lines:
        parts = line.split()
        prefix = f"{label_path}:{line_number}"
        if len(parts) != 5:
            issues.append(f"{prefix}: expected 5 fields, got {len(parts)}")
            continue
        try:
            values = [float(value) for value in parts]
        except ValueError:
            issues.append(f"{prefix}: all fields must be numeric")
            continue

        class_id, x_center, y_center, width, height = values
        if not all(math.isfinite(value) for value in values):
            issues.append(f"{prefix}: NaN/Inf values are not allowed")
            continue
        if not class_id.is_integer() or int(class_id) != 0:
            issues.append(f"{prefix}: class must be exactly 0 (wine_label), got {class_id}")
        if not all(0.0 <= value <= 1.0 for value in (x_center, y_center, width, height)):
            issues.append(f"{prefix}: x/y/w/h must be within 0..1")
        if width <= 0.0:
            issues.append(f"{prefix}: width must be > 0")
        if height <= 0.0:
            issues.append(f"{prefix}: height must be > 0")

        if not any(issue.startswith(prefix) for issue in issues):
            boxes.append((class_id, x_center, y_center, width, height))

    return boxes, issues, is_empty


def validate_yolo_dataset(dataset_root: Path = MANUAL_ANNOTATION_ROOT) -> dict[str, Any]:
    '''Validate train/val pairs and print an auditable summary table.'''
    dataset_root = Path(dataset_root)
    all_errors: list[str] = []
    all_warnings: list[str] = []
    summary_rows: list[dict[str, Any]] = []

    for split in ("train", "val"):
        images_dir = dataset_root / "images" / split
        labels_dir = dataset_root / "labels" / split
        images = list_images(images_dir)
        labels = sorted(labels_dir.rglob("*.txt")) if labels_dir.exists() else []

        image_stems = {path.relative_to(images_dir).with_suffix("") for path in images}
        label_stems = {path.relative_to(labels_dir).with_suffix("") for path in labels}
        orphan_labels = [path for path in labels if path.relative_to(labels_dir).with_suffix("") not in image_stems]

        missing_labels: list[Path] = []
        empty_labels: list[Path] = []
        invalid_rows = 0
        valid_boxes = 0

        for image_path in images:
            relative_image = image_path.relative_to(images_dir)
            label_path = labels_dir / relative_image.with_suffix(".txt")
            if not label_path.exists():
                missing_labels.append(image_path)
                continue
            boxes, issues, is_empty = _parse_yolo_label_file(label_path)
            valid_boxes += len(boxes)
            invalid_rows += len(issues)
            all_errors.extend(issues)
            if is_empty:
                empty_labels.append(label_path)

        if not images:
            all_errors.append(f"{split}: no supported images found in {images_dir}")
        all_errors.extend(f"{split}: missing label for {path}" for path in missing_labels)
        all_errors.extend(f"{split}: orphan label {path}" for path in orphan_labels)
        all_warnings.extend(f"{split}: empty label {path}" for path in empty_labels)

        summary_rows.append({
            "split": split,
            "images": len(images),
            "label_files": len(labels),
            "valid_boxes": valid_boxes,
            "missing_labels": len(missing_labels),
            "empty_labels": len(empty_labels),
            "orphan_labels": len(orphan_labels),
            "invalid_rows": invalid_rows,
        })

    summary = pd.DataFrame(summary_rows).set_index("split")
    display(summary)
    if all_errors:
        print(f"\nDATASET INVALID: {len(all_errors)} error(s). Fix these before training.")
        for issue in all_errors[:30]:
            print("ERROR:", issue)
        if len(all_errors) > 30:
            print(f"... {len(all_errors) - 30} more error(s) not shown")
    else:
        print("\nDataset structure and label rows are valid.")

    if all_warnings:
        print(f"\nWarnings: {len(all_warnings)}")
        for issue in all_warnings[:20]:
            print("WARNING:", issue)

    return {
        "ok": not all_errors,
        "summary": summary,
        "errors": all_errors,
        "warnings": all_warnings,
    }

In [ ]:
DATASET_REPORT = validate_yolo_dataset()

## 5. Visual check of annotations

Always inspect boxes before training. Green rectangles are ground truth; each image title shows its split. The function samples deterministically from train and validation data.

In [ ]:
def _normalized_box_to_xyxy(box: Iterable[float], image_width: int, image_height: int) -> tuple[float, float, float, float]:
    _, x_center, y_center, width, height = box
    x1 = (x_center - width / 2.0) * image_width
    y1 = (y_center - height / 2.0) * image_height
    x2 = (x_center + width / 2.0) * image_width
    y2 = (y_center + height / 2.0) * image_height
    return x1, y1, x2, y2


def visualize_random_annotations(n: int = 12, seed: int = SEED) -> None:
    candidates: list[tuple[str, Path, Path]] = []
    for split, images_dir, labels_dir in (
        ("train", TRAIN_IMAGES_DIR, TRAIN_LABELS_DIR),
        ("val", VAL_IMAGES_DIR, VAL_LABELS_DIR),
    ):
        for image_path in list_images(images_dir):
            label_path = labels_dir / image_path.relative_to(images_dir).with_suffix(".txt")
            if label_path.exists():
                candidates.append((split, image_path, label_path))

    if not candidates:
        print("No paired images and labels are available yet.")
        return

    sample_size = min(n, len(candidates))
    rng = random.Random(seed)
    selected = rng.sample(candidates, sample_size)
    columns = min(4, sample_size)
    rows = math.ceil(sample_size / columns)
    fig, axes = plt.subplots(rows, columns, figsize=(4.2 * columns, 4.2 * rows), squeeze=False)

    for axis, (split, image_path, label_path) in zip(axes.flat, selected):
        with Image.open(image_path) as opened_image:
            image = ImageOps.exif_transpose(opened_image).convert("RGB")
        width, height = image.size
        boxes, issues, _ = _parse_yolo_label_file(label_path)
        axis.imshow(image)
        for box in boxes:
            x1, y1, x2, y2 = _normalized_box_to_xyxy(box, width, height)
            rectangle = patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=2.0, edgecolor="#15803d", facecolor="none",
            )
            axis.add_patch(rectangle)
            axis.text(
                x1, max(0, y1 - 3), "wine_label",
                color="white", fontsize=9,
                bbox={"facecolor": "#15803d", "alpha": 0.9, "pad": 2, "edgecolor": "none"},
            )
        axis.set_title(f"{split}: {image_path.name}" + (" [INVALID LABEL]" if issues else ""))
        axis.axis("off")

    for axis in axes.flat[sample_size:]:
        axis.axis("off")
    fig.suptitle("Random YOLO annotations — verify tight label boundaries", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
visualize_random_annotations(n=12)

## 6. Training configuration

The primary model is `yolo11n.pt`. The defaults favor speed and stable Apple Silicon execution. Text-mirroring flips are disabled. Geometric and color augmentation are moderate and physically plausible; mosaic is kept low and disabled for the final epochs.

In [ ]:
if DEVICE_NAME == "cuda":
    default_batch = -1  # Ultralytics auto-batch on NVIDIA GPU
elif DEVICE_NAME == "mps":
    default_batch = 8   # conservative default for a 24 GB Apple Silicon machine
else:
    default_batch = 4

TRAIN_CONFIG = {
    "model": "yolo11n.pt",
    "imgsz": 640,
    "epochs": 50,
    "batch": default_batch,
    "patience": 10,
    "device": DEVICE,
    "workers": 4 if platform.system() == "Darwin" else 8,
    "project": str(RUNS_ROOT),
    "name": "train",
    "exist_ok": True,
    "pretrained": True,
    "optimizer": "auto",
    "seed": SEED,
    "deterministic": True,
    "single_cls": True,
    "plots": True,
    "save": True,
    "verbose": True,
    "amp": DEVICE_NAME == "cuda",
    "cache": False,
    # Conservative, physically plausible augmentation for front wine labels.
    "degrees": 5.0,
    "translate": 0.08,
    "scale": 0.25,
    "shear": 1.5,
    "perspective": 0.0005,
    "hsv_h": 0.015,
    "hsv_s": 0.40,
    "hsv_v": 0.25,
    "fliplr": 0.0,
    "flipud": 0.0,
    "mosaic": 0.20,
    "close_mosaic": 10,
    "mixup": 0.0,
    "copy_paste": 0.0,
}

VALIDATION_BATCH = 16 if TRAIN_CONFIG["batch"] == -1 else TRAIN_CONFIG["batch"]

pprint(TRAIN_CONFIG)

In [ ]:
def train_detector(config: dict[str, Any]) -> dict[str, Any]:
    '''Validate data, train YOLO11n, and return explicit run state.'''
    report = validate_yolo_dataset()
    if not report["ok"]:
        raise RuntimeError("Dataset validation failed. Fix reported errors before training.")

    config = dict(config)
    model_name = config.pop("model")
    model = YOLO(model_name)
    results = model.train(data=str(DATASET_YAML), **config)
    save_dir = Path(model.trainer.save_dir).resolve()
    return {"model": model, "results": results, "save_dir": save_dir}


TRAIN_STATE: dict[str, Any] = {}
TRAIN_METRICS: dict[str, float] = {}
VALIDATION_METRICS: dict[str, float] = {}
CROP_STATS: dict[str, Any] = {}
BENCHMARK_LATENCY: dict[str, Any] = {}

if RUN_TRAINING:
    TRAIN_STATE = train_detector(TRAIN_CONFIG)
    print("Training run saved to:", TRAIN_STATE["save_dir"])
else:
    print("Training skipped. Set RUN_TRAINING = True after dataset validation and visual inspection.")

## 7. Metrics, checkpoints, and training curves

After training, the best and last checkpoints are copied (never moved) into `models/yolo_label_detector/`. Validation reports Precision, Recall, mAP50, and mAP50–95.

In [ ]:
def extract_detection_metrics(metrics_object: Any) -> dict[str, float]:
    '''Extract the four core detection metrics across Ultralytics result versions.'''
    metrics: dict[str, float] = {}
    box = getattr(metrics_object, "box", None)
    if box is not None:
        mapping = {
            "precision": getattr(box, "mp", None),
            "recall": getattr(box, "mr", None),
            "mAP50": getattr(box, "map50", None),
            "mAP50-95": getattr(box, "map", None),
        }
        metrics.update({key: float(value) for key, value in mapping.items() if value is not None})

    results_dict = getattr(metrics_object, "results_dict", None)
    if isinstance(results_dict, dict):
        fallback_keys = {
            "precision": "metrics/precision(B)",
            "recall": "metrics/recall(B)",
            "mAP50": "metrics/mAP50(B)",
            "mAP50-95": "metrics/mAP50-95(B)",
        }
        for output_name, source_name in fallback_keys.items():
            if output_name not in metrics and source_name in results_dict:
                metrics[output_name] = float(results_dict[source_name])
    return metrics


def copy_training_checkpoints(train_save_dir: Path) -> dict[str, Optional[Path]]:
    weights_dir = Path(train_save_dir) / "weights"
    copied: dict[str, Optional[Path]] = {"best": None, "last": None}
    for checkpoint_name in ("best.pt", "last.pt"):
        source = weights_dir / checkpoint_name
        destination = MODEL_OUTPUT_DIR / checkpoint_name
        if source.exists():
            if source.resolve() != destination.resolve():
                shutil.copy2(source, destination)
            copied[checkpoint_name.removesuffix(".pt")] = destination.resolve()
            print(f"Copied {checkpoint_name} -> {destination}")
        else:
            print(f"Checkpoint not found: {source}")
    return copied


BEST_MODEL_PATH = MODEL_OUTPUT_DIR / "best.pt"
LAST_MODEL_PATH = MODEL_OUTPUT_DIR / "last.pt"

if TRAIN_STATE:
    TRAIN_METRICS = extract_detection_metrics(TRAIN_STATE["results"])
    copied_checkpoints = copy_training_checkpoints(TRAIN_STATE["save_dir"])
    BEST_MODEL_PATH = copied_checkpoints["best"] or BEST_MODEL_PATH
    LAST_MODEL_PATH = copied_checkpoints["last"] or LAST_MODEL_PATH
    print("Training metrics:")
    pprint(TRAIN_METRICS)
else:
    print("No training result in this kernel. Existing best checkpoint:", BEST_MODEL_PATH.exists())

In [ ]:
if RUN_VALIDATION:
    if not BEST_MODEL_PATH.exists():
        raise FileNotFoundError(f"Best checkpoint not found: {BEST_MODEL_PATH}")
    best_model = YOLO(str(BEST_MODEL_PATH))
    validation_result = best_model.val(
        data=str(DATASET_YAML),
        split="val",
        imgsz=TRAIN_CONFIG["imgsz"],
        batch=VALIDATION_BATCH,
        device=DEVICE,
        workers=TRAIN_CONFIG["workers"],
        plots=True,
        project=str(RUNS_ROOT),
        name="validation",
        exist_ok=True,
    )
    VALIDATION_METRICS = extract_detection_metrics(validation_result)
    print("Validation metrics:")
    pprint(VALIDATION_METRICS)
else:
    print("Validation skipped. Set RUN_VALIDATION = True when best.pt is available.")

In [ ]:
def show_training_curves(train_save_dir: Optional[Path] = None) -> None:
    candidate_paths: list[Path] = []
    if train_save_dir is not None:
        candidate_paths.append(Path(train_save_dir) / "results.png")
    candidate_paths.extend(sorted(RUNS_ROOT.rglob("results.png"), key=lambda path: path.stat().st_mtime, reverse=True))

    for path in candidate_paths:
        if path.exists():
            print("Training curves:", path)
            display(IPythonImage(filename=str(path)))
            return
    print("No results.png found yet. Ultralytics creates it after training when plots=True.")


show_training_curves(TRAIN_STATE.get("save_dir") if TRAIN_STATE else None)

## 8. Validation examples: ground truth vs prediction

Ground-truth boxes are solid green; predictions are dashed blue and include confidence. Hard cases are prioritized and titled as `FALSE NEGATIVE`, `LOW CONFIDENCE`, `POOR BBOX`, `MULTIPLE BOTTLES`, or `MULTIPLE DETECTIONS`. A multiple-detection flag is a review cue—it may be legitimate when multiple labeled bottles are present.

In [ ]:
def _prediction_records(result: Any) -> list[dict[str, float]]:
    if result.boxes is None or len(result.boxes) == 0:
        return []
    xyxy = result.boxes.xyxy.detach().cpu().numpy()
    confidences = result.boxes.conf.detach().cpu().numpy()
    classes = result.boxes.cls.detach().cpu().numpy().astype(int)
    records = []
    for coordinates, confidence, class_id in zip(xyxy, confidences, classes):
        if class_id == 0:
            records.append({
                "x1": float(coordinates[0]),
                "y1": float(coordinates[1]),
                "x2": float(coordinates[2]),
                "y2": float(coordinates[3]),
                "confidence": float(confidence),
            })
    return sorted(records, key=lambda row: row["confidence"], reverse=True)


def _bbox_iou(box_a: tuple[float, float, float, float], box_b: tuple[float, float, float, float]) -> float:
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    intersection_width = max(0.0, min(ax2, bx2) - max(ax1, bx1))
    intersection_height = max(0.0, min(ay2, by2) - max(ay1, by1))
    intersection = intersection_width * intersection_height
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - intersection
    return intersection / union if union > 0 else 0.0


def _classify_validation_case(
    ground_truth: list[tuple[float, float, float, float]],
    predictions: list[dict[str, float]],
    low_confidence_threshold: float = 0.5,
    poor_iou_threshold: float = 0.5,
) -> str:
    if ground_truth and (not predictions or len(predictions) < len(ground_truth)):
        return "FALSE NEGATIVE"
    if predictions and predictions[0]["confidence"] < low_confidence_threshold:
        return "LOW CONFIDENCE"
    if ground_truth and predictions:
        best_iou_per_gt = [
            max(
                _bbox_iou(gt_box, (pred["x1"], pred["y1"], pred["x2"], pred["y2"]))
                for pred in predictions
            )
            for gt_box in ground_truth
        ]
        if min(best_iou_per_gt) < poor_iou_threshold:
            return "POOR BBOX"
    if len(ground_truth) > 1:
        return "MULTIPLE BOTTLES"
    if len(predictions) > max(1, len(ground_truth)):
        return "MULTIPLE DETECTIONS"
    return "OK"


def visualize_validation_predictions(
    model_path: Path = BEST_MODEL_PATH,
    n: int = 12,
    confidence_floor: float = 0.01,
) -> None:
    model_path = Path(model_path)
    if not model_path.exists():
        print(f"Checkpoint not found: {model_path}")
        return
    validation_images = list_images(VAL_IMAGES_DIR)
    if not validation_images:
        print("No validation images found.")
        return

    detector = YOLO(str(model_path))
    cases: list[dict[str, Any]] = []
    for image_path in tqdm(validation_images, desc="Scoring validation images"):
        image_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if image_bgr is None:
            continue
        height, width = image_bgr.shape[:2]
        label_path = VAL_LABELS_DIR / image_path.relative_to(VAL_IMAGES_DIR).with_suffix(".txt")
        normalized_boxes, _, _ = _parse_yolo_label_file(label_path)
        ground_truth = [_normalized_box_to_xyxy(box, width, height) for box in normalized_boxes]
        result = detector.predict(
            source=image_bgr,
            imgsz=TRAIN_CONFIG["imgsz"],
            conf=confidence_floor,
            device=DEVICE,
            verbose=False,
        )[0]
        predictions = _prediction_records(result)
        status = _classify_validation_case(ground_truth, predictions)
        cases.append({
            "path": image_path,
            "image": cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB),
            "ground_truth": ground_truth,
            "predictions": predictions,
            "status": status,
        })

    severity = {
        "FALSE NEGATIVE": 0,
        "LOW CONFIDENCE": 1,
        "POOR BBOX": 2,
        "MULTIPLE BOTTLES": 3,
        "MULTIPLE DETECTIONS": 4,
        "OK": 5,
    }
    cases.sort(key=lambda row: (severity[row["status"]], row["path"].name))
    selected = cases[: min(n, len(cases))]
    if len(selected) < n:
        print(f"Only {len(selected)} validation image(s) are available; cannot show {n}.")

    columns = min(4, len(selected))
    rows = math.ceil(len(selected) / columns)
    fig, axes = plt.subplots(rows, columns, figsize=(4.5 * columns, 4.5 * rows), squeeze=False)
    for axis, case in zip(axes.flat, selected):
        axis.imshow(case["image"])
        for x1, y1, x2, y2 in case["ground_truth"]:
            axis.add_patch(patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=2.0, edgecolor="#15803d", facecolor="none",
            ))
            axis.text(x1, max(0, y1 - 3), "GT wine_label", color="white", fontsize=8,
                      bbox={"facecolor": "#15803d", "alpha": 0.9, "pad": 2, "edgecolor": "none"})
        for prediction in case["predictions"]:
            x1, y1, x2, y2 = (prediction[key] for key in ("x1", "y1", "x2", "y2"))
            axis.add_patch(patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=2.0, linestyle="--", edgecolor="#2563eb", facecolor="none",
            ))
            axis.text(x1, min(case["image"].shape[0] - 5, y2 + 12),
                      f"Pred {prediction['confidence']:.2f}", color="white", fontsize=8,
                      bbox={"facecolor": "#2563eb", "alpha": 0.9, "pad": 2, "edgecolor": "none"})
        title_color = "#b91c1c" if case["status"] != "OK" else "#1f2937"
        axis.set_title(f"{case['status']}\n{case['path'].name}", color=title_color)
        axis.axis("off")

    for axis in axes.flat[len(selected):]:
        axis.axis("off")
    fig.suptitle("Validation: ground truth (green) vs prediction (blue)", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
if RUN_VALIDATION_VISUALIZATION and BEST_MODEL_PATH.exists():
    visualize_validation_predictions(n=12)
elif RUN_VALIDATION_VISUALIZATION:
    print("Checkpoint is missing: train the model first.")
else:
    print("Validation visualization skipped. Set RUN_VALIDATION_VISUALIZATION = True when best.pt exists.")

## 9. Automatic label crop pipeline

The pipeline never moves or deletes source images. It selects the highest-confidence `wine_label` detection, adds configurable padding, clamps to image bounds, and writes:

- `successful/` for confidence ≥ 0.50;
- `low_confidence/` for 0.25 ≤ confidence < 0.50;
- a copy of the original into `failed/` for no detection, confidence < 0.25, or processing errors.

Low-confidence and failed cases always receive debug images when the source can be decoded. Successful debug images are controlled by `SAVE_DEBUG`.

In [ ]:
METADATA_COLUMNS = [
    "source_path",
    "source_filename",
    "crop_path",
    "status",
    "confidence",
    "x1",
    "y1",
    "x2",
    "y2",
    "padding",
    "image_width",
    "image_height",
]


def _stable_output_stem(source_path: Path, input_dir: Path) -> str:
    try:
        relative_text = source_path.resolve().relative_to(input_dir.resolve()).as_posix()
    except ValueError:
        relative_text = str(source_path.resolve())
    digest = hashlib.sha1(relative_text.encode("utf-8")).hexdigest()[:10]
    safe_stem = "".join(character if character.isalnum() or character in "-_" else "_" for character in source_path.stem)
    return f"{safe_stem}__{digest}"


def _synchronize_accelerator() -> None:
    if DEVICE_NAME == "cuda":
        torch.cuda.synchronize()
    elif DEVICE_NAME == "mps" and hasattr(torch, "mps"):
        torch.mps.synchronize()


def _draw_debug_image(
    image_bgr: np.ndarray,
    output_path: Path,
    box: Optional[tuple[int, int, int, int]],
    confidence: Optional[float],
    status: str,
) -> None:
    debug_image = image_bgr.copy()
    color = (30, 110, 240) if status != "successful" else (50, 170, 50)
    if box is not None:
        x1, y1, x2, y2 = box
        cv2.rectangle(debug_image, (x1, y1), (x2, y2), color, 3)
        label = f"wine_label {confidence:.3f}" if confidence is not None else "wine_label"
        cv2.putText(debug_image, label, (x1, max(24, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2, cv2.LINE_AA)
    cv2.putText(debug_image, status, (12, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2, cv2.LINE_AA)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(output_path), debug_image)


def crop_labels_from_directory(
    input_dir: Path,
    output_dir: Path,
    confidence_threshold: float = 0.5,
    low_confidence_threshold: float = 0.25,
    padding: float = 0.10,
    *,
    model_path: Path = BEST_MODEL_PATH,
    device: Any = DEVICE,
    save_debug: bool = SAVE_DEBUG,
    debug_dir: Path = PREDICTIONS_DEBUG_DIR,
    recursive: bool = True,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    '''Run robust directory inference and create crops, metadata, debug files, and a summary.'''
    input_dir = Path(input_dir).expanduser()
    output_dir = Path(output_dir)
    model_path = Path(model_path)
    debug_dir = Path(debug_dir)
    if not input_dir.exists() or not input_dir.is_dir():
        raise NotADirectoryError(f"Input directory does not exist: {input_dir}")
    if not model_path.exists():
        raise FileNotFoundError(f"Model checkpoint does not exist: {model_path}")
    if not (0.0 <= low_confidence_threshold <= confidence_threshold <= 1.0):
        raise ValueError("Require 0 <= low_confidence_threshold <= confidence_threshold <= 1")
    if padding < 0:
        raise ValueError("padding must be non-negative")

    status_directories = {
        "successful": output_dir / "successful",
        "low_confidence": output_dir / "low_confidence",
        "failed": output_dir / "failed",
    }
    for directory in [*status_directories.values(), debug_dir]:
        directory.mkdir(parents=True, exist_ok=True)

    detector = YOLO(str(model_path))
    image_paths = list_images(input_dir, recursive=recursive)
    metadata_rows: list[dict[str, Any]] = []
    error_log_path = output_dir / "crop_errors.log"
    errors: list[str] = []

    for source_path in tqdm(image_paths, desc="Cropping wine labels", unit="image"):
        output_stem = _stable_output_stem(source_path, input_dir)
        base_row: dict[str, Any] = {
            "source_path": str(source_path.resolve()),
            "source_filename": source_path.name,
            "crop_path": "",
            "status": "failed",
            "confidence": np.nan,
            "x1": np.nan,
            "y1": np.nan,
            "x2": np.nan,
            "y2": np.nan,
            "padding": float(padding),
            "image_width": np.nan,
            "image_height": np.nan,
        }
        image_bgr: Optional[np.ndarray] = None
        selected_box: Optional[tuple[int, int, int, int]] = None
        selected_confidence: Optional[float] = None

        try:
            image_bgr = cv2.imread(str(source_path), cv2.IMREAD_COLOR)
            if image_bgr is None:
                raise ValueError("OpenCV could not decode the image")
            image_height, image_width = image_bgr.shape[:2]
            base_row["image_width"] = int(image_width)
            base_row["image_height"] = int(image_height)

            prediction = detector.predict(
                source=image_bgr,
                imgsz=TRAIN_CONFIG["imgsz"],
                conf=0.001,
                device=device,
                verbose=False,
            )[0]
            detections = _prediction_records(prediction)

            if detections:
                selected = max(detections, key=lambda item: item["confidence"])
                selected_confidence = float(selected["confidence"])
                raw_x1, raw_y1, raw_x2, raw_y2 = (selected[key] for key in ("x1", "y1", "x2", "y2"))
                box_width = max(0.0, raw_x2 - raw_x1)
                box_height = max(0.0, raw_y2 - raw_y1)
                x1 = max(0, int(math.floor(raw_x1 - box_width * padding)))
                y1 = max(0, int(math.floor(raw_y1 - box_height * padding)))
                x2 = min(image_width, int(math.ceil(raw_x2 + box_width * padding)))
                y2 = min(image_height, int(math.ceil(raw_y2 + box_height * padding)))
                selected_box = (x1, y1, x2, y2)
                base_row.update({
                    "confidence": selected_confidence,
                    "x1": x1,
                    "y1": y1,
                    "x2": x2,
                    "y2": y2,
                })

            if selected_confidence is not None and selected_confidence >= confidence_threshold:
                status = "successful"
            elif selected_confidence is not None and selected_confidence >= low_confidence_threshold:
                status = "low_confidence"
            else:
                status = "failed"
            base_row["status"] = status

            if status in {"successful", "low_confidence"}:
                if selected_box is None:
                    raise RuntimeError("Internal error: accepted detection has no bounding box")
                x1, y1, x2, y2 = selected_box
                crop = image_bgr[y1:y2, x1:x2]
                if crop.size == 0:
                    raise ValueError("Predicted crop is empty after boundary clamping")
                crop_path = status_directories[status] / f"{output_stem}.jpg"
                if not cv2.imwrite(str(crop_path), crop, [cv2.IMWRITE_JPEG_QUALITY, 95]):
                    raise IOError(f"Could not write crop: {crop_path}")
                base_row["crop_path"] = str(crop_path.resolve())
            else:
                failed_path = status_directories["failed"] / f"{output_stem}{source_path.suffix.lower()}"
                shutil.copy2(source_path, failed_path)
                base_row["crop_path"] = str(failed_path.resolve())

            if save_debug or status in {"low_confidence", "failed"}:
                _draw_debug_image(
                    image_bgr,
                    debug_dir / f"{output_stem}.jpg",
                    selected_box,
                    selected_confidence,
                    status,
                )

        except Exception as exc:
            base_row["status"] = "failed"
            error_text = f"{source_path}\t{type(exc).__name__}: {exc}"
            errors.append(error_text)
            failed_path = status_directories["failed"] / f"{output_stem}{source_path.suffix.lower()}"
            try:
                shutil.copy2(source_path, failed_path)
                base_row["crop_path"] = str(failed_path.resolve())
            except Exception as copy_exc:
                errors.append(f"{source_path}\tfailed-copy: {copy_exc}")
            if image_bgr is not None:
                _draw_debug_image(
                    image_bgr,
                    debug_dir / f"{output_stem}.jpg",
                    selected_box,
                    selected_confidence,
                    "failed",
                )
        metadata_rows.append(base_row)

    metadata = pd.DataFrame(metadata_rows, columns=METADATA_COLUMNS)
    metadata_path = output_dir / "crops_metadata.csv"
    temporary_metadata_path = output_dir / "crops_metadata.tmp.csv"
    metadata.to_csv(temporary_metadata_path, index=False)
    temporary_metadata_path.replace(metadata_path)

    if errors:
        error_log_path.write_text("\n".join(errors) + "\n", encoding="utf-8")
    elif error_log_path.exists():
        error_log_path.unlink()

    status_counts = metadata["status"].value_counts().to_dict() if not metadata.empty else {}
    finite_confidences = pd.to_numeric(metadata["confidence"], errors="coerce").dropna()
    summary = {
        "total": int(len(metadata)),
        "successful": int(status_counts.get("successful", 0)),
        "low_confidence": int(status_counts.get("low_confidence", 0)),
        "failed": int(status_counts.get("failed", 0)),
        "average_confidence": float(finite_confidences.mean()) if not finite_confidences.empty else None,
        "metadata_csv": str(metadata_path.resolve()),
        "error_log": str(error_log_path.resolve()) if errors else None,
    }
    print("Crop summary:")
    pprint(summary)
    return metadata, summary

### 8.1 Small inference-input run

Copy a few unlabeled photos into `datasets/yolo_label_detector/inference_input/`, set `RUN_SAMPLE_CROP = True`, and run this cell before attempting the 45k-image job.

In [ ]:
if RUN_SAMPLE_CROP:
    sample_metadata, CROP_STATS = crop_labels_from_directory(
        input_dir=INFERENCE_INPUT_DIR,
        output_dir=CROPS_ROOT,
        confidence_threshold=0.50,
        low_confidence_threshold=0.25,
        padding=0.10,
        model_path=BEST_MODEL_PATH,
        save_debug=SAVE_DEBUG,
    )
    display(sample_metadata.head(20))
else:
    print("Sample crop skipped. Set RUN_SAMPLE_CROP = True after best.pt is available.")

### 8.2 Bulk processing for the large image collection

Set `BULK_INPUT_DIR` in the parameters cell to the actual source directory. The function recursively supports JPG, JPEG, PNG, and WebP, shows `tqdm` progress, isolates per-file errors, and writes a complete CSV summary. The original collection is never modified.

In [ ]:
bulk_path_is_configured = str(BULK_INPUT_DIR) not in {"", ".", "..."}

if RUN_BULK_CROP:
    if not bulk_path_is_configured:
        raise ValueError("Set BULK_INPUT_DIR to the real source directory first.")
    bulk_metadata, CROP_STATS = crop_labels_from_directory(
        input_dir=BULK_INPUT_DIR,
        output_dir=CROPS_ROOT,
        confidence_threshold=0.50,
        low_confidence_threshold=0.25,
        padding=0.10,
        model_path=BEST_MODEL_PATH,
        save_debug=SAVE_DEBUG,
        recursive=True,
    )
    display(pd.DataFrame([CROP_STATS]))
else:
    print("Bulk crop skipped. Configure BULK_INPUT_DIR and set RUN_BULK_CROP = True when ready.")

## 10. Iterative annotation / active learning

Recommended loop:

1. Manually annotate 100–150 images.
2. Train YOLO11n.
3. Run it on new raw images.
4. Select low-confidence, failed, and geometrically difficult cases.
5. Correct those cases manually and add them to train/validation without leakage.
6. Retrain.
7. Repeat until recall and box quality are stable.

The helper below copies the hardest source images into a new annotation batch and creates a manifest. It does not delete or move originals.

In [ ]:
def select_active_learning_candidates(
    metadata_csv: Path = CROPS_ROOT / "crops_metadata.csv",
    destination_dir: Path = RAW_IMAGES_DIR / "active_learning_batch",
    n: int = 100,
) -> pd.DataFrame:
    metadata_csv = Path(metadata_csv)
    destination_dir = Path(destination_dir)
    if not metadata_csv.exists():
        raise FileNotFoundError(f"Metadata CSV not found: {metadata_csv}")
    if n <= 0:
        raise ValueError("n must be positive")

    metadata = pd.read_csv(metadata_csv)
    required_columns = {"source_path", "status", "confidence"}
    missing_columns = required_columns.difference(metadata.columns)
    if missing_columns:
        raise ValueError(f"Metadata is missing columns: {sorted(missing_columns)}")

    candidates = metadata[metadata["status"].isin(["failed", "low_confidence"])].copy()
    candidates["status_priority"] = candidates["status"].map({"failed": 0, "low_confidence": 1})
    candidates["confidence_sort"] = pd.to_numeric(candidates["confidence"], errors="coerce").fillna(-1.0)
    candidates = candidates.sort_values(
        ["status_priority", "confidence_sort", "source_path"],
        ascending=[True, True, True],
    ).head(n)

    destination_dir.mkdir(parents=True, exist_ok=True)
    manifest_rows: list[dict[str, Any]] = []
    for row in candidates.itertuples(index=False):
        source_path = Path(row.source_path)
        if not source_path.exists():
            continue
        output_stem = _stable_output_stem(source_path, source_path.parent)
        destination_path = destination_dir / f"{output_stem}{source_path.suffix.lower()}"
        shutil.copy2(source_path, destination_path)
        manifest_rows.append({
            "source_path": str(source_path),
            "annotation_copy": str(destination_path.resolve()),
            "status": row.status,
            "confidence": row.confidence,
        })

    manifest = pd.DataFrame(manifest_rows)
    manifest_path = destination_dir / "active_learning_manifest.csv"
    manifest.to_csv(manifest_path, index=False)
    print(f"Selected {len(manifest)} image(s) -> {destination_dir}")
    print(f"Manifest: {manifest_path}")
    return manifest


# Example (run after crop metadata exists):
# active_learning_batch = select_active_learning_candidates(n=100)
# display(active_learning_batch.head(20))

## 11. Inference benchmark

The benchmark warms up with 10 images and then performs at least 100 single-image predictions. It reports:

- Ultralytics model inference time (`result.speed['inference']`);
- end-to-end wall time including image decode, preprocessing, inference, best-box selection, padding, and in-memory crop.

CUDA/MPS synchronization is applied around wall-clock measurements.

In [ ]:
def _latency_summary(milliseconds: list[float]) -> dict[str, float]:
    values = np.asarray(milliseconds, dtype=float)
    return {
        "average_ms": float(np.mean(values)),
        "p50_ms": float(np.percentile(values, 50)),
        "p95_ms": float(np.percentile(values, 95)),
    }


def benchmark_detector(
    model_path: Path = BEST_MODEL_PATH,
    image_dir: Path = VAL_IMAGES_DIR,
    warmup: int = 10,
    predictions: int = 100,
    padding: float = 0.10,
) -> dict[str, Any]:
    model_path = Path(model_path)
    image_paths = list_images(Path(image_dir))
    if not model_path.exists():
        raise FileNotFoundError(f"Model checkpoint not found: {model_path}")
    if not image_paths:
        raise FileNotFoundError(f"No benchmark images found in: {image_dir}")

    detector = YOLO(str(model_path))
    prediction_count = max(100, int(predictions))

    for index in tqdm(range(warmup), desc="Benchmark warm-up"):
        image_bgr = cv2.imread(str(image_paths[index % len(image_paths)]), cv2.IMREAD_COLOR)
        if image_bgr is None:
            continue
        detector.predict(source=image_bgr, imgsz=TRAIN_CONFIG["imgsz"], conf=0.001, device=DEVICE, verbose=False)
    _synchronize_accelerator()

    inference_times_ms: list[float] = []
    full_pipeline_times_ms: list[float] = []

    for index in tqdm(range(prediction_count), desc="Benchmark predictions"):
        image_path = image_paths[index % len(image_paths)]
        _synchronize_accelerator()
        start = time.perf_counter()
        image_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if image_bgr is None:
            continue
        result = detector.predict(
            source=image_bgr,
            imgsz=TRAIN_CONFIG["imgsz"],
            conf=0.001,
            device=DEVICE,
            verbose=False,
        )[0]
        detections = _prediction_records(result)
        if detections:
            selected = detections[0]
            image_height, image_width = image_bgr.shape[:2]
            box_width = selected["x2"] - selected["x1"]
            box_height = selected["y2"] - selected["y1"]
            x1 = max(0, int(selected["x1"] - box_width * padding))
            y1 = max(0, int(selected["y1"] - box_height * padding))
            x2 = min(image_width, int(selected["x2"] + box_width * padding))
            y2 = min(image_height, int(selected["y2"] + box_height * padding))
            _ = image_bgr[y1:y2, x1:x2].copy()
        _synchronize_accelerator()
        full_pipeline_times_ms.append((time.perf_counter() - start) * 1000.0)

        inference_time = getattr(result, "speed", {}).get("inference")
        if inference_time is not None:
            inference_times_ms.append(float(inference_time))

    if not full_pipeline_times_ms:
        raise RuntimeError("No benchmark predictions completed successfully.")

    benchmark = {
        "device": DEVICE_NAME,
        "imgsz": TRAIN_CONFIG["imgsz"],
        "warmup_predictions": int(warmup),
        "measured_predictions": len(full_pipeline_times_ms),
        "model_inference": _latency_summary(inference_times_ms) if inference_times_ms else None,
        "full_preprocess_inference_crop": _latency_summary(full_pipeline_times_ms),
    }
    pprint(benchmark)
    return benchmark


if RUN_BENCHMARK:
    benchmark_image_dir = VAL_IMAGES_DIR if list_images(VAL_IMAGES_DIR) else INFERENCE_INPUT_DIR
    BENCHMARK_LATENCY = benchmark_detector(
        model_path=BEST_MODEL_PATH,
        image_dir=benchmark_image_dir,
        warmup=10,
        predictions=100,
    )
else:
    print("Benchmark skipped. Set RUN_BENCHMARK = True after best.pt and test images are available.")

## 12. Final handoff summary

This cell prints only results available in the current kernel. Missing values mean the corresponding opt-in stage has not been run yet—not that a zero score was observed.

In [ ]:
def _print_result_block(title: str, value: Any) -> None:
    print(f"\n{title}")
    print("-" * len(title))
    if value:
        pprint(value)
    else:
        print("Not available in this kernel yet.")


print("Annotated source dataset:", SOURCE_ANNOTATED_DATASET_ROOT.resolve())
print("Prepared one-class dataset:", PREPARED_DATASET_ROOT.resolve())
print("Annotated source dataset:", SOURCE_ANNOTATED_DATASET_ROOT.resolve())
print("Prepared one-class dataset:", PREPARED_DATASET_ROOT.resolve())
print("Annotated source dataset:", SOURCE_ANNOTATED_DATASET_ROOT.resolve())
print("Prepared one-class dataset:", PREPARED_DATASET_ROOT.resolve())
print("Best YOLO model path:", BEST_MODEL_PATH.resolve())
print("Best model exists:", BEST_MODEL_PATH.exists())
_print_result_block("Train metrics", TRAIN_METRICS)
_print_result_block("Validation metrics", VALIDATION_METRICS)
_print_result_block("Crop statistics", CROP_STATS)
_print_result_block("Benchmark latency", BENCHMARK_LATENCY)

print("\nRecommendations")
print("---------------")
print("If recall is low -> annotate missed/failed examples.")
print("If bounding boxes are poor -> add difficult geometry and correct box boundaries.")
print("If recall is high and boxes are stable -> bulk-crop the 45k images and train DINO.")